# Módulo geografía
Ciudades, distritos judiciales y departamentos. El literal del PDF no se toca:
estas tablas solo alimentan columnas derivadas.

In [ ]:
import re
import unicodedata

import pandas as pd

DEPARTAMENTOS = ["Chuquisaca", "La Paz", "Cochabamba", "Oruro", "Potosí",
                 "Tarija", "Santa Cruz", "Beni", "Pando"]

# Ciudad capital (cuadro 9.1.3) -> departamento. El Alto es la única que no es capital de departamento.
CIUDAD_A_DEPARTAMENTO = {
    "SUCRE": "Chuquisaca",
    "LA PAZ": "La Paz",
    "EL ALTO": "La Paz",
    "COCHABAMBA": "Cochabamba",
    "ORURO": "Oruro",
    "POTOSI": "Potosí",
    "TARIJA": "Tarija",
    "SANTA CRUZ": "Santa Cruz",
    "TRINIDAD": "Beni",
    "COBIJA": "Pando",
}

# Distrito judicial -> departamento. OFICINA NACIONAL / NACIONAL es la administración central, no un territorio.
DISTRITO_A_DEPARTAMENTO = {
    "OFICINA NACIONAL": None,
    "NACIONAL": None,
    "CHUQUISACA": "Chuquisaca",
    "LA PAZ": "La Paz",
    "COCHABAMBA": "Cochabamba",
    "ORURO": "Oruro",
    "POTOSI": "Potosí",
    "TARIJA": "Tarija",
    "SANTA CRUZ": "Santa Cruz",
    "BENI": "Beni",
    "PANDO": "Pando",
}

NOMBRE_DEPARTAMENTO = {
    "CHUQUISACA": "Chuquisaca",
    "LA PAZ": "La Paz",
    "COCHABAMBA": "Cochabamba",
    "ORURO": "Oruro",
    "POTOSI": "Potosí",
    "TARIJA": "Tarija",
    "SANTA CRUZ": "Santa Cruz",
    "BENI": "Beni",
    "PANDO": "Pando",
}

CAPITALES_NORM = set(CIUDAD_A_DEPARTAMENTO.keys())
PROVINCIAS_NORM = set()
for k in DISTRITO_A_DEPARTAMENTO:
    if DISTRITO_A_DEPARTAMENTO[k] is not None:
        PROVINCIAS_NORM.add(k)

# Nombres que solo pueden ser ciudad o solo distrito: resuelven páginas con título vacío.
CAPITALES_INEQUIVOCAS = {"SUCRE", "EL ALTO", "TRINIDAD", "COBIJA"}
PROVINCIAS_INEQUIVOCAS = {"CHUQUISACA", "BENI", "PANDO"}

# Totales que en el PDF no repiten el nombre de la entidad: (cuadro, página, ámbito, entidad, rótulo).
ALIASES_TOTAL_TERRITORIAL = {
    ("5.3.1.3", 356, "capital", "TRINIDAD", "TOTAL BENI"),
    ("5.3.1.3", 356, "capital", "COBIJA", "TOTAL PANDO"),
    ("6.3.1.4", 359, "capital", "TRINIDAD", "TOTAL BENI"),
    ("6.3.1.4", 359, "capital", "COBIJA", "TOTAL PANDO"),
    ("6.1.1.2", 409, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
    ("6.1.2.1", 449, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
    ("6.1.3.3", 524, "provincia", "PANDO", "TOTAL COBIJA"),
    ("6.3.1.1", 607, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
    ("6.3.1.2", 610, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
    ("6.3.1.3", 613, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
    ("6.3.1.4", 616, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
    ("6.3.1.5", 619, "provincia", "CHUQUISACA", "TOTAL SUCRE"),
}

In [ ]:
def es_faltante(valor):
    if valor is None:
        return True
    try:
        if bool(pd.isna(valor)):
            return True
    except (TypeError, ValueError):
        pass
    return isinstance(valor, str) and not valor.strip()


def normalizar_geografia(valor):
    # Mayúsculas, sin tildes, sin puntuación y sin espacios dobles.
    if es_faltante(valor):
        return None
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("ascii").upper()
    texto = " ".join(re.sub(r"[^A-Z0-9]+", " ", texto).split())
    if texto == "":
        return None
    return texto


def departamento_de_ciudad(ciudad):
    return CIUDAD_A_DEPARTAMENTO.get(normalizar_geografia(ciudad))


def departamento_de_distrito(distrito):
    return DISTRITO_A_DEPARTAMENTO.get(normalizar_geografia(distrito))


def departamento_normalizado(nombre):
    return NOMBRE_DEPARTAMENTO.get(normalizar_geografia(nombre))


def total_corresponde_a_entidad(cuadro, pagina, ambito, entidad, rotulo_total):
    entidad_norm = normalizar_geografia(entidad)
    rotulo_norm = normalizar_geografia(rotulo_total)
    if entidad_norm is None or rotulo_norm is None:
        return False
    if rotulo_norm == "TOTAL " + entidad_norm:
        return True
    if es_faltante(cuadro) or es_faltante(pagina) or es_faltante(ambito):
        return False
    try:
        pagina_norm = int(pagina)
    except (TypeError, ValueError, OverflowError):
        return False
    clave = (str(cuadro).strip(), pagina_norm, str(ambito).strip().lower(), entidad_norm, rotulo_norm)
    return clave in ALIASES_TOTAL_TERRITORIAL


def departamento_segun_ambito(ambito, ciudad=None, distrito=None):
    if ambito == "capital":
        return departamento_de_ciudad(ciudad)
    if ambito == "provincia":
        return departamento_de_distrito(distrito)
    return None


def ambito_de_contexto(titulo_pagina, entidades):
    # El número de cuadro no decide el ámbito: 6.3.1.4 (p. 357-359) es del capítulo 6
    # pero su encabezado dice "Ciudades Capitales y El Alto".
    titulo = normalizar_geografia(titulo_pagina)
    if titulo is None:
        titulo = ""
    if "CIUDADES CAPITALES" in titulo and "EL ALTO" in titulo:
        return "capital"
    if "PROVINCI" in titulo:
        return "provincia"
    claves = set()
    for e in entidades:
        n = normalizar_geografia(e)
        if n is not None:
            claves.add(n)
    es_capital = len(claves & CAPITALES_INEQUIVOCAS) > 0
    es_provincia = len(claves & PROVINCIAS_INEQUIVOCAS) > 0
    if es_capital != es_provincia:
        if es_capital:
            return "capital"
        return "provincia"
    return None